In [1]:
import pandas as pd
import numpy as np

# Read raw data
metrics = pd.read_csv("metrics.csv")
vendor = pd.read_csv("Vendor.csv")

# Create synthetic complaint text based on defect quantity and downtime
def generate_complaint_text(row):
    defect_qty = row["Defect Qty"]
    downtime = row["Downtime min"]

    if downtime >= 100 and defect_qty >= 100000:
        return "Critical repeated defect caused production line stop and delayed corrective action."
    
    elif downtime >= 100:
        return "Production downtime occurred due to supplier quality issue and late response."
    
    elif defect_qty >= 100000:
        return "High defect quantity reported with recurring material quality issue."
    
    elif defect_qty >= 50000:
        return "Moderate defect quantity identified during quality inspection."
    
    else:
        return "Minor defect identified during routine quality inspection."

metrics["complaint_text"] = metrics.apply(generate_complaint_text, axis=1)

print(metrics[["Vendor ID", "Defect Qty", "Downtime min", "complaint_text"]].head(10))

   Vendor ID  Defect Qty  Downtime min  \
0          2           0            60   
1          2           0            60   
2          2           1            60   
3         59           9            10   
4         46          47            30   
5         16       20009           218   
6          4           1            75   
7         62         570            70   
8         62         623            30   
9        103       14418             0   

                                      complaint_text  
0  Minor defect identified during routine quality...  
1  Minor defect identified during routine quality...  
2  Minor defect identified during routine quality...  
3  Minor defect identified during routine quality...  
4  Minor defect identified during routine quality...  
5  Production downtime occurred due to supplier q...  
6  Minor defect identified during routine quality...  
7  Minor defect identified during routine quality...  
8  Minor defect identified during routine 

In [2]:
risk_keywords = {
    "critical": 25,
    "repeated": 20,
    "recurring": 20,
    "line stop": 25,
    "production downtime": 20,
    "delayed corrective action": 20,
    "late response": 15,
    "safety": 25,
    "root cause": 15,
    "customer escalation": 25
}

def calculate_text_risk_score(text):
    text = text.lower()
    score = 0

    for keyword, weight in risk_keywords.items():
        if keyword in text:
            score += weight

    return min(score, 100)

metrics["text_risk_score"] = metrics["complaint_text"].apply(calculate_text_risk_score)

metrics["high_risk_text_flag"] = metrics["text_risk_score"].apply(
    lambda score: 1 if score >= 70 else 0
)

print(metrics[["complaint_text", "text_risk_score", "high_risk_text_flag"]].head(10))

                                      complaint_text  text_risk_score  \
0  Minor defect identified during routine quality...                0   
1  Minor defect identified during routine quality...                0   
2  Minor defect identified during routine quality...                0   
3  Minor defect identified during routine quality...                0   
4  Minor defect identified during routine quality...                0   
5  Production downtime occurred due to supplier q...               35   
6  Minor defect identified during routine quality...                0   
7  Minor defect identified during routine quality...                0   
8  Minor defect identified during routine quality...                0   
9  Minor defect identified during routine quality...                0   

   high_risk_text_flag  
0                    0  
1                    0  
2                    0  
3                    0  
4                    0  
5                    0  
6                    

In [3]:
text_risk_summary = (
    metrics
    .groupby("Vendor ID")
    .agg(
        complaint_count=("complaint_text", "count"),
        avg_text_risk_score=("text_risk_score", "mean"),
        high_risk_text_count=("high_risk_text_flag", "sum")
    )
    .reset_index()
)

text_risk_summary["avg_text_risk_score"] = text_risk_summary["avg_text_risk_score"].round(2)

print(text_risk_summary.head(10))

   Vendor ID  complaint_count  avg_text_risk_score  high_risk_text_count
0          1              710                 3.20                     0
1          2              323                 3.19                     0
2          3               10                 0.00                     0
3          4              396                 0.91                     0
4          5               16                 4.38                     0
5          6               22                 4.77                     0
6          7               46                 0.00                     0
7          8               35                 0.00                     0
8          9              404                 1.13                     0
9         10                1                 0.00                     0


In [5]:
supplier_risk_summary = pd.read_csv("supplier_risk_summary.csv")

supplier_risk_summary_nlp = supplier_risk_summary.merge(
    text_risk_summary,
    on="Vendor ID",
    how="left"
)

print(supplier_risk_summary_nlp.head())

   Vendor ID  total_defect_qty  total_downtime_minutes  total_defect_reports  \
0          1            484594                   26185                   710   
1          2           3838867                   10405                   323   
2          4           3169402                    5978                   396   
3         36           3988201                    2300                   118   
4         11           2982348                    3088                   187   

        Vendor  downtime_risk_score  defect_qty_risk_score  \
0      Reddoit           100.000000              12.150692   
1      Plustax            39.736490              96.255605   
2    Quotelane            22.829864              79.469465   
3  Solholdings             8.783655             100.000000   
4    Dentocity            11.793011              74.779280   

   report_frequency_risk_score  supplier_risk_score   risk_level  \
0                   100.000000                69.25  Medium Risk   
1         

In [6]:
supplier_risk_summary_nlp["supplier_risk_score_nlp"] = (
    supplier_risk_summary_nlp["downtime_risk_score"] * 0.30
    + supplier_risk_summary_nlp["defect_qty_risk_score"] * 0.30
    + supplier_risk_summary_nlp["report_frequency_risk_score"] * 0.20
    + supplier_risk_summary_nlp["avg_text_risk_score"] * 0.20
)

supplier_risk_summary_nlp["supplier_risk_score_nlp"] = (
    supplier_risk_summary_nlp["supplier_risk_score_nlp"].round(2)
)

In [14]:
q70 = supplier_risk_summary_nlp["supplier_risk_score_nlp"].quantile(0.70)
q30 = supplier_risk_summary_nlp["supplier_risk_score_nlp"].quantile(0.30)

def assign_risk_level_nlp(score):
    if score >= q70:
        return "High Risk"
    elif score >= q30:
        return "Medium Risk"
    else:
        return "Low Risk"
supplier_risk_summary_nlp["risk_level_nlp"] = (
    supplier_risk_summary_nlp["supplier_risk_score_nlp"].apply(assign_risk_level_nlp)
)

def generate_recommendation_nlp(row):
    if row["risk_level_nlp"] == "High Risk" and row["high_risk_text_count"] > 0:
        return "Prioritize supplier audit, review repeated issues, and request corrective action plan."
    
    elif row["risk_level_nlp"] == "High Risk":
        return "Prioritize supplier review and corrective action plan."
    
    elif row["risk_level_nlp"] == "Medium Risk" and row["avg_text_risk_score"] >= 50:
        return "Monitor supplier closely and investigate text-based risk signals."
    
    elif row["risk_level_nlp"] == "Medium Risk":
        return "Monitor supplier performance and investigate recurring issues."
    
    else:
        return "Maintain regular monitoring."

supplier_risk_summary_nlp["recommendation_nlp"] = supplier_risk_summary_nlp.apply(
    generate_recommendation_nlp,
    axis=1
)

In [16]:
supplier_risk_summary_nlp = supplier_risk_summary_nlp.sort_values(
    by="supplier_risk_score_nlp",
    ascending=False
)

supplier_risk_summary_nlp.to_csv(
    "supplier_risk_summary_nlp.csv",
    index=False
)

print("supplier_risk_summary_nlp.csv has been created successfully.")
print(supplier_risk_summary_nlp.head(10))

supplier_risk_summary_nlp.csv has been created successfully.
    Vendor ID  total_defect_qty  total_downtime_minutes  total_defect_reports  \
0           1            484594                   26185                   710   
1           2           3838867                   10405                   323   
2           4           3169402                    5978                   396   
3          36           3988201                    2300                   118   
4          11           2982348                    3088                   187   
5           9            735527                   10006                   404   
6          20           2589319                    4215                    96   
7          16           2005374                    3327                   152   
8          62            513653                   10275                   170   
11         34           1681683                    3382                    70   

         Vendor  downtime_risk_score  defect_qt